# 03. 평가 지표와 비교 주장 감사하기

목표: UI-JEPA 결과를 split별로 비교하고, 논문의 Intent Similarity가 raw metric의 단순 평균만으로 재현되지 않는다는 점과 작은 zero-shot sample의 불확실성을 확인합니다.

In [ ]:
rows = {
    'IIW few UI-JEPA 384': ([66.51, 66.33, 42.94, 65.48], 64.50),
    'IIT few UI-JEPA OCR': ([87.43, 83.73, 69.17, 81.51], 82.03),
}

for name, (raw_metrics, reported_intent_similarity) in rows.items():
    plain_average = sum(raw_metrics) / len(raw_metrics)
    print(name)
    print('  raw arithmetic mean:', round(plain_average, 2))
    print('  reported Intent Sim.:', reported_intent_similarity)
    print('  gap:', round(reported_intent_similarity - plain_average, 2))
    assert abs(reported_intent_similarity - plain_average) > 0.1

print('결론: 논문의 정규화 절차가 없으면 Intent Similarity를 raw 표 값만으로 재현할 수 없습니다.')

In [ ]:
intent_similarity = {
    'IIW few': {'UI-JEPA': 64.50, 'Claude': 64.76, 'GPT-4 Turbo': 63.36},
    'IIW zero': {'UI-JEPA': 52.16, 'Claude': 60.35, 'GPT-4 Turbo': 58.24},
    'IIT few + OCR': {'UI-JEPA': 82.03, 'Claude': 61.95, 'GPT-4 Turbo': 59.70},
    'IIT zero + OCR': {'UI-JEPA': 34.99, 'Claude': 56.82, 'GPT-4 Turbo': 52.69},
}

for split, scores in intent_similarity.items():
    ranking = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    print(f'{split:15s}:', ' > '.join(f'{name} {score}' for name, score in ranking))

assert max(intent_similarity['IIT few + OCR'], key=intent_similarity['IIT few + OCR'].get) == 'UI-JEPA'
assert max(intent_similarity['IIT zero + OCR'], key=intent_similarity['IIT zero + OCR'].get) == 'Claude'

In [ ]:
from random import Random

# 원 논문의 per-example score는 없으므로 n=45인 IIT zero-shot의 불확실성을
# 가상의 binary 성공 18개로만 설명합니다. 논문 confidence interval이 아닙니다.
outcomes = [1] * 18 + [0] * 27
rng = Random(4081)
bootstrap_means = []
for _ in range(5000):
    sample = [rng.choice(outcomes) for _ in outcomes]
    bootstrap_means.append(sum(sample) / len(sample))
bootstrap_means.sort()
low = bootstrap_means[int(0.025 * len(bootstrap_means))]
high = bootstrap_means[int(0.975 * len(bootstrap_means))]
print(f'illustrative success rate={sum(outcomes)/len(outcomes):.1%}, bootstrap interval≈[{low:.1%}, {high:.1%}]')
assert high - low > 0.2

In [ ]:
claims = {
    'compute cost reduction': 50.5,
    'latency improvement': 6.6,
    'OCR latency multiplier': 1.135,
}
for label, value in claims.items():
    print(label, value)
print('감사 체크: 동일 hardware/API 시점, frame 수, batch, refusal 포함 여부, energy와 memory를 기록하세요.')

## 결론

평균 하나로는 familiar few-shot 강점과 unfamiliar zero-shot 약점을 동시에 표현할 수 없습니다. 재현 보고서에는 dataset·split·OCR 여부별 원 지표, per-example bootstrap interval, 실패·refusal을 포함한 분모, 동일한 frame budget을 함께 제시해야 합니다.